# Technical QC exploratory notebook

KneadData and MultiQC summary for the salivary WGS meta-analysis.

This notebook is kept as an exploratory QC file. It is not part of the main modeling pipeline.

In [ ]:
# Setup
from __future__ import annotations

import re
import warnings

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

from config import (
    DEFAULT_METADATA_PATH,
    QC_CLEAN_DATA_DIR,
    QC_MULTIQC_BASE,
    QC_RESULTS_DIR,
    QC_FIGURES_DIR,
    RANDOM_STATE,
)
from common import read_csv_with_run

warnings.filterwarnings("ignore")

QC_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
QC_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"KneadData logs: {QC_CLEAN_DATA_DIR}")
print(f"MultiQC base: {QC_MULTIQC_BASE}")
print(f"Metadata: {DEFAULT_METADATA_PATH}")
print(f"QC output: {QC_RESULTS_DIR}")

## Parse KneadData logs

In [ ]:
def parse_kneaddata_log(log_path) -> dict:
    """Parse one KneadData log into sample-level QC metrics."""
    text = log_path.read_text(errors="ignore")
    sample = log_path.stem.replace(".kneaddata", "")
    bioproject = log_path.parent.name
    is_paired = "paired_1.fastq" in text or "paired1.fastq" in text

    input_matches = re.findall(r"([\d,]+) reads; of these:", text)
    reads_input = int(input_matches[-1].replace(",", "")) if input_matches else None

    final_matches = re.findall(
        r"READ COUNT: final \S+\s*:.*?:\s*([\d.]+)",
        text,
        re.IGNORECASE,
    )
    reads_microbial = int(sum(float(x) for x in final_matches)) if final_matches else None

    human_matches = re.findall(r"Total contaminate sequences in file.*?:\s*([\d.]+)", text)
    reads_human = int(sum(float(x) for x in human_matches)) if human_matches else None
    if reads_human is None and reads_input and reads_microbial:
        reads_human = reads_input - reads_microbial

    pct_human = None
    if reads_input and reads_microbial is not None:
        pct_human = round((1 - reads_microbial / reads_input) * 100, 2)

    bowtie_matches = re.findall(r"([\d.]+)% overall alignment rate", text)

    return {
        "run": sample,
        "bioproject": bioproject,
        "library_type": "paired" if is_paired else "single",
        "reads_input": reads_input,
        "reads_human": reads_human,
        "reads_microbial": reads_microbial,
        "pct_human": pct_human,
        "pct_aligned_bt2": float(bowtie_matches[-1]) if bowtie_matches else None,
        "n_final_lines": len(final_matches),
    }


records = []
for log_path in sorted(QC_CLEAN_DATA_DIR.rglob("*.kneaddata.log")):
    try:
        records.append(parse_kneaddata_log(log_path))
    except Exception as exc:
        print(f"[WARN] Could not parse {log_path.name}: {exc}")

df_qc = pd.DataFrame(records)
if df_qc.empty:
    raise RuntimeError(f"No KneadData log records were parsed from {QC_CLEAN_DATA_DIR}")

print(f"Samples: {len(df_qc)}")
print(f"Bioprojects: {df_qc['bioproject'].nunique()}")
print("Missing values by field:")
display(df_qc.isna().sum().to_frame("missing"))

df_qc.head()

## Merge metadata and MultiQC statistics

In [ ]:
def make_disease_label(row: pd.Series) -> str:
    has_k05 = row.get("K05_gingivitis_periodontitis", 0) == 1
    has_k02 = row.get("K02_caries", 0) == 1
    has_healthy = row.get("healthy", 0) == 1

    if has_k05 and has_k02:
        return "mixed"
    if has_k05:
        return "periodontitis"
    if has_k02:
        return "caries"
    if has_healthy:
        return "healthy"
    return "unknown"


metadata = read_csv_with_run(DEFAULT_METADATA_PATH)

df = df_qc.merge(
    metadata.drop(columns=["bioproject"], errors="ignore"),
    on="run",
    how="left",
)

df["disease_label"] = df.apply(make_disease_label, axis=1)

print(f"Samples after metadata merge: {len(df)}")
print("Disease labels:")
display(df["disease_label"].value_counts().to_frame("n"))

df.head()

In [ ]:
multiqc_frames = []
for bp_dir in sorted(QC_MULTIQC_BASE.iterdir()):
    stats_path = bp_dir / "multiqc_data" / "multiqc_general_stats.txt"
    if not stats_path.exists():
        continue

    tmp = pd.read_csv(stats_path, sep="	")
    tmp["bioproject"] = bp_dir.name
    multiqc_frames.append(tmp)

if not multiqc_frames:
    raise RuntimeError(f"No MultiQC general_stats files were found under {QC_MULTIQC_BASE}")

df_mqc_raw = pd.concat(multiqc_frames, ignore_index=True)

df_mqc = df_mqc_raw[
    ~df_mqc_raw["Sample"].str.contains(r"contam|repeats\.removed", regex=True, na=False)
].copy()

df_mqc["run"] = (
    df_mqc["Sample"]
    .str.replace(r"_(paired|unmatched)_\d$", "", regex=True)
    .str.replace(r"_R?[12]$", "", regex=True)
)

df_mqc_agg = (
    df_mqc.groupby(["run", "bioproject"])
    .agg(
        gc=("fastqc-percent_gc", "mean"),
        pct_dup=("fastqc-percent_duplicates", "mean"),
        avg_len=("fastqc-avg_sequence_length", "mean"),
        total_seqs=("fastqc-total_sequences", "sum"),
    )
    .reset_index()
)

df_master = df.merge(
    df_mqc_agg.drop(columns=["bioproject"], errors="ignore"),
    on="run",
    how="left",
)

print(f"Master table samples: {len(df_master)}")
print("Missing values in MultiQC fields:")
display(df_master[["gc", "pct_dup", "avg_len", "total_seqs"]].isna().sum().to_frame("missing"))

df_master.head()

## Summary tables

In [ ]:
ct = pd.crosstab(df_master["bioproject"], df_master["disease_label"])
display(ct)

single_group_projects = ct[ct.gt(0).sum(axis=1) == 1]
if not single_group_projects.empty:
    print("Bioprojects represented by a single clinical group:")
    display(single_group_projects)

In [ ]:
summary = (
    df_master.groupby("bioproject")
    .agg(
        n=("run", "count"),
        reads_micro_median=("reads_microbial", "median"),
        reads_micro_min=("reads_microbial", "min"),
        reads_micro_max=("reads_microbial", "max"),
        pct_human_median=("pct_human", "median"),
        gc_median=("gc", "median"),
        pct_dup_median=("pct_dup", "median"),
        avg_len_median=("avg_len", "median"),
        n_healthy=("disease_label", lambda x: (x == "healthy").sum()),
        n_periodontitis=("disease_label", lambda x: (x == "periodontitis").sum()),
        n_caries=("disease_label", lambda x: (x == "caries").sum()),
        n_mixed=("disease_label", lambda x: (x == "mixed").sum()),
    )
    .round(1)
)

summary["likely_pre_decontaminated"] = summary["pct_human_median"] < 5

df_master.to_csv(QC_RESULTS_DIR / "qc_metadata.csv", index=False)
summary.to_csv(QC_RESULTS_DIR / "summary_bioproject_qc.csv")

print(f"Saved: {QC_RESULTS_DIR / 'qc_metadata.csv'}")
print(f"Saved: {QC_RESULTS_DIR / 'summary_bioproject_qc.csv'}")

display(summary)

## Final QC panel

In [ ]:
GROUP_COLORS = {
    "healthy": "#6BAE8A",
    "periodontitis": "#8D7CB4",
    "caries": "#D4875A",
    "mixed": "#c0a0d0",
    "unknown": "#cccccc",
}

GROUP_ORDER = ["healthy", "periodontitis", "caries", "mixed", "unknown"]
GROUP_LABELS = {
    "healthy": "Healthy",
    "periodontitis": "Periodontitis (K05)",
    "caries": "Caries (K02)",
    "mixed": "K05 + K02",
    "unknown": "Unknown",
}

bp_total = df_master.groupby("bioproject").size().sort_values(ascending=False)
bp_order = bp_total.index.tolist()
bp_order_box = (
    df_master.groupby("bioproject")["reads_microbial"]
    .median()
    .sort_values(ascending=False)
    .index
    .tolist()
)

ct = pd.crosstab(df_master["bioproject"], df_master["disease_label"])
for group in GROUP_ORDER:
    if group not in ct.columns:
        ct[group] = 0

ct = ct.reindex(index=bp_order, fill_value=0)
present_groups = [group for group in GROUP_ORDER if ct[group].sum() > 0]

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.7,
    "xtick.major.width": 0.7,
    "ytick.major.width": 0.7,
    "svg.fonttype": "none",
})

In [ ]:
def draw_bioproject_boxplot(ax, column, ylabel, bp_order_box, yscale=None, annotation=None):
    data = [
        df_master.loc[df_master["bioproject"] == bp, column].dropna().values
        for bp in bp_order_box
    ]
    x_positions = np.arange(len(bp_order_box))
    box_palette = sns.color_palette("Set2", n_colors=len(bp_order_box))

    box = ax.boxplot(
        data,
        positions=x_positions,
        patch_artist=True,
        widths=0.52,
        medianprops={"color": "black", "linewidth": 1.2},
        whiskerprops={"linewidth": 0.8, "color": "#555555"},
        capprops={"linewidth": 0.8, "color": "#555555"},
        boxprops={"linewidth": 0.7},
        flierprops={"marker": ""},
        showfliers=False,
    )

    for patch, color in zip(box["boxes"], box_palette):
        patch.set_facecolor(color)
        patch.set_alpha(0.78)

    rng = np.random.default_rng(RANDOM_STATE)
    for x_i, values in enumerate(data):
        if len(values) == 0:
            continue
        jitter = rng.uniform(-0.20, 0.20, size=len(values))
        ax.scatter(
            x_positions[x_i] + jitter,
            values,
            color="black",
            alpha=0.25,
            s=5,
            linewidths=0,
            zorder=3,
        )

    if yscale:
        ax.set_yscale(yscale)

    if annotation:
        ax.text(
            0.02,
            0.97,
            annotation,
            transform=ax.transAxes,
            fontsize=6.8,
            va="top",
            color="#555555",
            bbox={
                "boxstyle": "round,pad=0.3",
                "facecolor": "white",
                "edgecolor": "#cccccc",
                "alpha": 0.85,
            },
        )

    ax.set_xticks(x_positions)
    ax.set_xticklabels(bp_order_box, rotation=40, ha="right", fontsize=7.5, style="italic")
    ax.set_xlim(-0.7, len(bp_order_box) - 0.3)
    ax.set_ylabel(ylabel, fontsize=8.5)
    ax.tick_params(axis="y", labelsize=8)

In [ ]:
fig = plt.figure(figsize=(18, 10))

gs_outer = gridspec.GridSpec(
    2,
    1,
    figure=fig,
    height_ratios=[1.05, 1],
    hspace=0.50,
    left=0.06,
    right=0.97,
    top=0.95,
    bottom=0.10,
)

gs_top = gridspec.GridSpecFromSubplotSpec(
    1,
    2,
    subplot_spec=gs_outer[0],
    width_ratios=[1.5, 1],
    wspace=0.28,
)

gs_bottom = gridspec.GridSpecFromSubplotSpec(
    1,
    3,
    subplot_spec=gs_outer[1],
    wspace=0.36,
)

ax_bar = fig.add_subplot(gs_top[0])
ax_microbial = fig.add_subplot(gs_top[1])
ax_length = fig.add_subplot(gs_bottom[0])
ax_gc = fig.add_subplot(gs_bottom[1])
ax_duplicates = fig.add_subplot(gs_bottom[2])

y_positions = np.arange(len(bp_order))
left = np.zeros(len(bp_order))

for group in present_groups:
    values = ct[group].values.astype(float)
    ax_bar.barh(
        y_positions,
        values,
        left=left,
        color=GROUP_COLORS[group],
        height=0.62,
        linewidth=0.25,
        edgecolor="white",
        label=GROUP_LABELS[group],
    )

    for y_i, (value, left_value) in enumerate(zip(values, left)):
        if value >= 8:
            ax_bar.text(
                left_value + value / 2,
                y_i,
                str(int(value)),
                ha="center",
                va="center",
                fontsize=7,
                color="white",
                fontweight="bold",
            )

    left += values

for y_i, bioproject in enumerate(bp_order):
    ax_bar.text(
        left[y_i] + 1.5,
        y_i,
        f"n = {int(bp_total[bioproject])}",
        ha="left",
        va="center",
        fontsize=7.5,
        color="#444444",
    )

ax_bar.set_yticks(y_positions)
ax_bar.set_yticklabels(bp_order, fontsize=8.5, style="italic")
ax_bar.set_xlabel("Number of samples", fontsize=9)
ax_bar.set_xlim(0, left.max() * 1.20)
ax_bar.invert_yaxis()
ax_bar.legend(
    fontsize=7.8,
    frameon=False,
    bbox_to_anchor=(0, -0.13),
    loc="upper left",
    ncol=len(present_groups),
    handlelength=1.0,
    columnspacing=0.9,
)

draw_bioproject_boxplot(
    ax_microbial,
    "reads_microbial",
    "Reads (log scale)",
    bp_order_box,
    yscale="log",
)
ax_microbial.yaxis.set_major_formatter(
    mticker.FuncFormatter(
        lambda x, _: (
            f"{x / 1e6:.0f}M"
            if x >= 1e6
            else f"{int(x / 1e3)}K"
            if x >= 1e3
            else str(int(x))
        )
    )
)

draw_bioproject_boxplot(
    ax_length,
    "avg_len",
    "bp",
    bp_order_box,
    annotation=(
        "Illumina HiSeq 2000, PE100 -> ~98 bp\n"
        "Illumina HiSeq, PE150 -> ~149-151 bp\n"
        "Illumina HiSeq 2500, PE250 -> ~242 bp\n"
        "Ion Torrent PGM -> ~200 bp"
    ),
)
for y_line in [100, 150, 250]:
    ax_length.axhline(y_line, color="#aaaaaa", linestyle=":", linewidth=0.8, alpha=0.6)

draw_bioproject_boxplot(ax_gc, "gc", "GC %", bp_order_box)
ax_gc.set_ylim(30, 65)

draw_bioproject_boxplot(ax_duplicates, "pct_dup", "Duplicates %", bp_order_box)

figure_path = QC_FIGURES_DIR / "technical_qc_cohort_panel.svg"
fig.savefig(figure_path, bbox_inches="tight", facecolor="white")

fig.canvas.draw()
renderer = fig.canvas.get_renderer()
axes_to_save = {
    "cohort_composition": ax_bar,
    "microbial_reads": ax_microbial,
    "read_length": ax_length,
    "gc_content": ax_gc,
    "duplicate_rate": ax_duplicates,
}

for name, axis in axes_to_save.items():
    bbox = axis.get_tightbbox(renderer).expanded(1.04, 1.08)
    bbox = bbox.transformed(fig.dpi_scale_trans.inverted())
    panel_path = QC_FIGURES_DIR / f"technical_qc_cohort_panel_{name}.svg"
    fig.savefig(panel_path, bbox_inches=bbox, facecolor="white")

print(f"Saved: {figure_path}")
for name in axes_to_save:
    print(f"Saved: {QC_FIGURES_DIR / f'technical_qc_cohort_panel_{name}.svg'}")

plt.show()